# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Clone fork GitHub của bạn vào Colab.
REPO_URL = "https://github.com/VuQuocHuy89/K4-Track-4-Day10-VisionLanguageModel.git"
import os
import subprocess
from pathlib import Path

repo = Path("/content/K4-Track-4-Day10-VisionLanguageModel")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
os.chdir(repo)
subprocess.run([os.sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt", "pytest"], check=True)
subprocess.run([os.sys.executable, "-m", "pytest", "-q"], check=True)
import torch
assert torch.cuda.is_available(), "Chọn Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))


In [ ]:
# Chỉ lưu Kaggle API trong Colab Secrets, không ghi khóa vào notebook.
from google.colab import drive, userdata

STUDENT_ID = "2A202602929"
kaggle_username = userdata.get("KAGGLE_USERNAME")
kaggle_key = userdata.get("KAGGLE_KEY")
assert kaggle_username and kaggle_key, "Thêm KAGGLE_USERNAME và KAGGLE_KEY vào Colab Secrets"
os.environ["KAGGLE_USERNAME"] = kaggle_username
os.environ["KAGGLE_KEY"] = kaggle_key
drive.mount("/content/drive")
DRIVE_ROOT = f"/content/drive/MyDrive/day10_{STUDENT_ID}/checkpoints"
from lab_work import sync_outputs
sync_outputs(DRIVE_ROOT, restore=True)


## 1. Dữ liệu: tải và chia split

In [ ]:
# Tải trực tiếp trên Colab và chia dữ liệu bằng script gốc của đề.
if not Path("data/splits/meta.json").exists():
    subprocess.run([os.sys.executable, "scripts/prepare_data.py", "--kaggle"], check=True)


In [ ]:
from IPython.display import display
from PIL import Image
from lab_work import check_data, draw_boxes
from scripts.labcommon import read_jsonl

meta = check_data()
train_rows = read_jsonl("data/splits/train.jsonl")
for row in train_rows[:6]:
    with Image.open(Path("data") / row["image_path"]) as image:
        print(row["image_id"], [obj["label"] for obj in row["objects"]])
        display(draw_boxes(image.convert("RGB"), row["objects"]))


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from lab_work import PROMPT, objects_to_answer, self_check, yolo_to_box

self_check()
print(PROMPT)
print(objects_to_answer(train_rows[0]["objects"]))


## 3. E0: zero-shot

In [ ]:
import gc
import torch
from lab_work import load_model, load_processor, predict, score

processor = load_processor()
if not Path("report/evidence/e0.json").exists():
    model = load_model()
    predict(model, processor, "test", "work/e0.jsonl")
    del model
    gc.collect()
    torch.cuda.empty_cache()
    score("E0", "work/e0.jsonl")
    sync_outputs(DRIVE_ROOT)


In [ ]:
# Xem bằng chứng E0 và vài output thô để phân tích lỗi trong REPORT.md.
import json
e0 = json.loads(Path("report/evidence/e0.json").read_text(encoding="utf-8"))
print("E0 F1:", e0["micro"]["f1"], "CI95:", e0["f1_ci95"],
      "parse_valid_rate:", e0["parse_valid_rate"])
for row in read_jsonl("work/e0.jsonl")[:5]:
    print(row["image_id"], repr(row["raw"][:500]))


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from lab_work import collate, encode_example

sample = encode_example(train_rows[0], processor)
visual_tokens = int(sample["image_grid_thw"][0].prod()) // 4
answer_tokens = sample["input_ids"][sample["labels"] != -100]
print("Vision tokens:", visual_tokens)
print("Assistant answer:", processor.tokenizer.decode(answer_tokens))
assert visual_tokens == 220
assert objects_to_answer(train_rows[0]["objects"]) in processor.tokenizer.decode(answer_tokens)
batch = collate([sample, encode_example(train_rows[1], processor)], processor.tokenizer.pad_token_id)
assert batch["input_ids"].shape == batch["labels"].shape


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from lab_work import apply_setup

# apply_setup kiểm tra đúng 252 LoRA modules cho E1/E2 và 348 cho E3 khi nạp model.


## 6. Vòng huấn luyện

In [ ]:
from lab_work import load_checkpoint, run_trained, save_checkpoint, train

# run_trained huấn luyện 300 bước, lưu adapter/projector mỗi 100 bước,
# nạp lại checkpoint để suy luận và ghi metadata thật vào evidence.


## 7. Suy luận và chấm E1–E3

In [ ]:
# Chạy lần lượt. Nếu Colab ngắt, chạy lại cell này; các evidence đã xong được bỏ qua.
for setup in ("E1", "E2", "E3"):
    evidence = Path(f"report/evidence/{setup.lower()}.json")
    if evidence.exists():
        print(setup, "already scored")
        continue
    result = run_trained(setup, setup, processor, drive_root=DRIVE_ROOT)
    print(setup, "F1:", result["micro"]["f1"], "CI95:", result["f1_ci95"])


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4: chỉ đổi lượng dữ liệu train (10% so với 100%); các tham số khác giữ nguyên.
from lab_work import run_trained, save_examples, score

results = {setup: json.loads(Path(f"report/evidence/{setup.lower()}.json").read_text(encoding="utf-8"))
           for setup in ("E1", "E2", "E3")}
BASE_SETUP = max(results, key=lambda setup: results[setup]["micro"]["f1"])
print("Base setup:", BASE_SETUP)
if not Path("report/evidence/e4_data_0.1.json").exists():
    run_trained("E4_data_0.1", BASE_SETUP, processor, train_fraction=0.1, drive_root=DRIVE_ROOT)
if not Path("report/evidence/e4_data_1.0.json").exists():
    score("E4_data_1.0", f"work/{BASE_SETUP.lower()}.jsonl",
          results[BASE_SETUP]["meta"])
    sync_outputs(DRIVE_ROOT)
save_examples(BASE_SETUP)
subprocess.run([os.sys.executable, "scripts/check_submission.py"], check=False)
# Sau khi chạy, tải report/evidence và report/figures từ Colab về repo cá nhân.
import shutil
archive = shutil.make_archive(
    f"/content/drive/MyDrive/day10_{STUDENT_ID}_report", "zip",
    root_dir=".", base_dir="report",
)
print("Report archive:", archive)
